<a href="https://colab.research.google.com/github/slrico/Log-clickstream-Analysis/blob/main/WindowsLogAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- This dataset consists of system event logs recorded over a specific period, primarily capturing informational messages related to system components and operations.
- It includes detailed entries such as timestamps, event levels, component identifiers, and descriptions of specific activities like loading servicing stacks and initializing system processes.
- This dataset serves as a valuable resource for analyzing system behavior, monitoring system updates, or troubleshooting issues by providing a chronological record of key system events.

In [1]:
import pandas as pd
from tabulate import tabulate
import json
import re

Maindata = pd.read_csv('./Windows_2k.log_structured.csv')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None)
Maindata = pd.read_csv('./Windows_2k.log_structured.csv')
Maindata.head()


,LineId,Date,Time,Level,Component,Content,EventId,EventTemplate
0,1,2016-09-28,04:30:30,Info,CBS,Loaded Servicing Stack v6.1.7601.23505 with Core: C:\Windows\winsxs\amd64_microsoft-windows-servicingstack_31bf3856ad364e35_6.1.7601.23505_none_681aa442f6fed7f0\cbscore.dll,E23,Loaded Servicing Stack <*> with Core: <*>\cbscore.dll
1,2,2016-09-28,04:30:31,Info,CSI,00000001@2016/9/27:20:30:31.455 WcpInitialize (wcp.dll version 0.0.0.6) called (stack @0x7fed806eb5d @0x7fef9fb9b6d @0x7fef9f8358f @0xff83e97c @0xff83d799 @0xff83db2f),E13,<*>@<*>/<*>/<*>:<*>:<*>:<*>.<*> WcpInitialize (wcp.dll version <*>) called (stack @<*>)
2,3,2016-09-28,04:30:31,Info,CSI,00000002@2016/9/27:20:30:31.458 WcpInitialize (wcp.dll version 0.0.0.6) called (stack @0x7fed806eb5d @0x7fefa006ade @0x7fef9fd2984 @0x7fef9f83665 @0xff83e97c @0xff83d799),E13,<*>@<*>/<*>/<*>:<*>:<*>:<*>.<*> WcpInitialize (wcp.dll version <*>) called (stack @<*>)
3,4,2016-09-28,04:30:31,Info,CSI,00000003@2016/9/27:20:30:31.458 WcpInitialize (wcp.dll version 0.0.0.6) called (stack @0x7fed806eb5d @0x7fefa1c8728 @0x7fefa1c8856 @0xff83e474 @0xff83d7de @0xff83db2f),E13,<*>@<*>/<*>/<*>:<*>:<*>:<*>.<*> WcpInitialize (wcp.dll version <*>) called (stack @<*>)
4,5,2016-09-28,04:30:31,Info,CBS,Ending TrustedInstaller initialization.,E17,Ending TrustedInstaller initialization.


### Dataset Columns Items

- LineId: A unique identifier for each log entry.

- Date: The date when the event occurred.

- Time: The exact time of the event.

- Level: The severity or type of the event (e.g., "Info" indicates informational messages).

- Component: The system component responsible for the event. For example:

- CBS: Refers to the Component-Based Servicing, which manages Windows updates and installations.

- CSI: Refers to the Component Servicing Infrastructure, which supports the CBS in managing system components.

- Content: A detailed description of the event. For instance:

- The CBS entry indicates that a servicing stack was loaded, specifying the version and file path.

- The CSI entry logs the initialization of a component (e.g., wcp.dll), including a stack trace.

- EventId: A unique identifier for the type of event.

- EventTemplate: A generalized template for the event description, where placeholders (e.g., <*>) represent variable content.

In [3]:

limited_data = Maindata[:4]
print(tabulate(limited_data, headers='keys', tablefmt='psql'))

+----+----------+------------+----------+---------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+-----------------------------------------------------------------------------------------+
|    |   LineId | Date       | Time     | Level   | Component   | Content                                                                                                                                                                      | EventId   | EventTemplate                                                                           |
|----+----------+------------+----------+---------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+--------------------------------------------------------------

In [4]:
# Convert to JSON format
Maindata = pd.read_csv('./Windows_2k.log_structured.csv')

# Set pandas display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', 100)  # Adjust the column width
pd.set_option('display.expand_frame_repr', False)  # Disable wrapping

# Display the DataFrame in a readable format
#print(Maindata.to_string(index=False))

json_data = Maindata.to_json(orient='records', lines=True)

# Save to file
# with open('output.json', 'w') as f:
#    f.write(json_data)
# print("Data has been converted to JSON format and saved as 'output.json'.")

In [5]:
# Limit the DataFrame to 4 rows
limited_data = Maindata.head(4)
json_result = limited_data.to_json(orient='records', lines=False)
print(json_result)

[{"LineId":1,"Date":"2016-09-28","Time":"04:30:30","Level":"Info","Component":"CBS","Content":"Loaded Servicing Stack v6.1.7601.23505 with Core: C:\\Windows\\winsxs\\amd64_microsoft-windows-servicingstack_31bf3856ad364e35_6.1.7601.23505_none_681aa442f6fed7f0\\cbscore.dll","EventId":"E23","EventTemplate":"Loaded Servicing Stack <*> with Core: <*>\\cbscore.dll"},{"LineId":2,"Date":"2016-09-28","Time":"04:30:31","Level":"Info","Component":"CSI","Content":"00000001@2016\/9\/27:20:30:31.455 WcpInitialize (wcp.dll version 0.0.0.6) called (stack @0x7fed806eb5d @0x7fef9fb9b6d @0x7fef9f8358f @0xff83e97c @0xff83d799 @0xff83db2f)","EventId":"E13","EventTemplate":"<*>@<*>\/<*>\/<*>:<*>:<*>:<*>.<*> WcpInitialize (wcp.dll version <*>) called (stack @<*>)"},{"LineId":3,"Date":"2016-09-28","Time":"04:30:31","Level":"Info","Component":"CSI","Content":"00000002@2016\/9\/27:20:30:31.458 WcpInitialize (wcp.dll version 0.0.0.6) called (stack @0x7fed806eb5d @0x7fefa006ade @0x7fef9fd2984 @0x7fef9f83665 @0xff

In [ ]:
print(Maindata.describe(include='all'))  # Summary for all columns

             LineId        Date      Time Level Component                                           Content EventId                                               EventTemplate
count   2000.000000        2000      2000  2000      2000                                              2000    2000                                                        2000
unique          NaN           2        76     1         2                                               963      50                                                          50
top             NaN  2016-09-29  02:03:48  Info       CBS  Warning: Unrecognized packageExtended attribute.     E36  Session: <*>_<*> initialized by client WindowsUpdateAgent.
freq            NaN        1047       210  2000      1973                                               280     608                                                         608
mean    1000.500000         NaN       NaN   NaN       NaN                                               NaN     NaN     

## Log File Cleaning and Preprocessing

In [6]:
# Convert JSON string back to a list of dictionaries
data_list = json.loads(json_result)

# Define a cleaning function
def clean_content(text):
    cleaned_text = text.strip()  # Remove leading/trailing whitespace
    # Replace multiple spaces with a single space
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
    cleaned_text = ''.join(char for char in cleaned_text if char.isalnum() or char.isspace())  # Remove special characters
    cleaned_text = cleaned_text.lower()
    return cleaned_text

# Iterate through each item and clean the Content field
for item in data_list:
    item['Content'] = clean_content(item['Content'])
    item['EventTemplate'] = clean_content(item['EventTemplate'])

limited_data_list = data_list[:4]  # Keep only the first 4 elements
cleaned_json_result = json.dumps(limited_data_list, indent=4)  # Use indent for readability
print(cleaned_json_result)

[
    {
        "LineId": 1,
        "Date": "2016-09-28",
        "Time": "04:30:30",
        "Level": "Info",
        "Component": "CBS",
        "Content": "loaded servicing stack v61760123505 with core cwindowswinsxsamd64microsoftwindowsservicingstack31bf3856ad364e3561760123505none681aa442f6fed7f0cbscoredll",
        "EventId": "E23",
        "EventTemplate": "loaded servicing stack  with core cbscoredll"
    },
    {
        "LineId": 2,
        "Date": "2016-09-28",
        "Time": "04:30:31",
        "Level": "Info",
        "Component": "CSI",
        "Content": "000000012016927203031455 wcpinitialize wcpdll version 0006 called stack 0x7fed806eb5d 0x7fef9fb9b6d 0x7fef9f8358f 0xff83e97c 0xff83d799 0xff83db2f",
        "EventId": "E13",
        "EventTemplate": " wcpinitialize wcpdll version  called stack "
    },
    {
        "LineId": 3,
        "Date": "2016-09-28",
        "Time": "04:30:31",
        "Level": "Info",
        "Component": "CSI",
        "Content": "0000000220

In [ ]:
def preprocess_and_convert_to_json(data_list, max_rows=4):
    """
    Preprocess the data list, limit rows, and convert it to formatted JSON.

    Parameters:
        data_list (list): List of dictionaries containing the data.
        max_rows (int): Maximum number of rows to include in the final JSON output.

    Returns:
        str: Formatted JSON string.
    """
    df = pd.DataFrame(data_list)
    df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
    df.drop(columns=['Date', 'Time'], inplace=True)
    df['Content_Length'] = df['Content'].str.len()
    df = pd.get_dummies(df, columns=['Level', 'Component'], drop_first=True)
    df.fillna('', inplace=True)
    limited_df = df.head(max_rows)
    cleaned_json_result = json.dumps(json.loads(limited_df.to_json(orient='records', lines=False)), indent=4)
    return cleaned_json_result

result = preprocess_and_convert_to_json(data_list, max_rows=4)
print(result)

[
    {
        "LineId": 1,
        "Content": "loaded servicing stack v61760123505 with core cwindowswinsxsamd64microsoftwindowsservicingstack31bf3856ad364e3561760123505none681aa442f6fed7f0cbscoredll",
        "EventId": "E23",
        "EventTemplate": "loaded servicing stack  with core cbscoredll",
        "Datetime": 1475037030000,
        "Content_Length": 152,
        "Component_CSI": false
    },
    {
        "LineId": 2,
        "Content": "000000012016927203031455 wcpinitialize wcpdll version 0006 called stack 0x7fed806eb5d 0x7fef9fb9b6d 0x7fef9f8358f 0xff83e97c 0xff83d799 0xff83db2f",
        "EventId": "E13",
        "EventTemplate": " wcpinitialize wcpdll version  called stack ",
        "Datetime": 1475037031000,
        "Content_Length": 146,
        "Component_CSI": true
    },
    {
        "LineId": 3,
        "Content": "000000022016927203031458 wcpinitialize wcpdll version 0006 called stack 0x7fed806eb5d 0x7fefa006ade 0x7fef9fd2984 0x7fef9f83665 0xff83e97c 0xff83d79

## Normalizing each column with it's own regex
- Pre processing of data enteries

In [ ]:
import re

log_entries = [
    # Add your log entries here as strings
]

def parse_log_entry(entry):
    # Use regex to extract the relevant parts
    match = re.match(
        r'\| (\d+) \| (\d+) \| (.+?) \| (.+?) \| (.+?) \| (.+?) \| (.+?) \| (.+?) \|',
        entry)
    if match:
        return {
            'index': int(match.group(1)),
            'unknown_col': match.group(2),  # Depending on what this data is
            'date': match.group(3),
            'time': match.group(4),
            'log_level': match.group(5),
            'source': match.group(6),
            'message': match.group(7),
            'event_id': match.group(8) if match.lastindex > 7 else None,
        }
    return None

parsed_logs = [parse_log_entry(entry) for entry in log_entries if entry]


In [ ]:
import re

log_entries = [
    # Add your log entries here as strings
]

pattern = r'^\| (\d+) \| (\d+) \| (\d{4}-\d{2}-\d{2}) \| (\d{2}:\d{2}:\d{2}) \| (.+?) \| (.+?) \| ([\d\w@.\s(),]+) \| ([\w\d]+) \| (.+)$'

def normalize_log_entry(entry):
    match = re.match(pattern, entry)
    if match:
        return f"| <*> | <*> | {match.group(3)} | {match.group(4)} | {match.group(5)} | {match.group(6)} | <*> | {match.group(8)} | {match.group(9)}"
    return entry  # Return the entry unchanged if it doesn't match

normalized_logs = [normalize_log_entry(entry) for entry in log_entries]
for log in normalized_logs:
    print(log)


**Anomaly Detection Models**

- Unusual data patterns:
- Isolation Forest or Local Outlier Factor for detecting anomalies.

**Recommendation Systems**
- content or components based on user patterns:
- Collaborative Filtering or Content-Based Filtering methods.